# 0.0. Setup

In [22]:
!pip install -qq playwright
!playwright install chromium
!playwright install-deps chromium

Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

In [25]:
import asyncio
from playwright.async_api import async_playwright
import os
import re
import time
from datetime import datetime

In [26]:
!ls -lhsa

total 540K
4.0K drwxr-xr-x 1 root root 4.0K Mar 19 08:16 .
4.0K drwxr-xr-x 1 root root 4.0K Mar 19 07:06 ..
4.0K drwxr-xr-x 4 root root 4.0K Mar 17 17:58 .config
 88K -rw-r--r-- 1 root root  86K Mar 19 08:15 debug_failed_BoSY_2025-26_Assessment_Results_attempt1.png
 88K -rw-r--r-- 1 root root  86K Mar 19 08:15 debug_failed_BoSY_2025-26_Assessment_Results_attempt2.png
 88K -rw-r--r-- 1 root root  86K Mar 19 08:15 debug_failed_BoSY_2025-26_Assessment_Results_attempt3.png
 84K -rw-r--r-- 1 root root  84K Mar 19 08:16 debug_failed_EoSY_2025-26_Assessment_Results_attempt1.png
 84K -rw-r--r-- 1 root root  84K Mar 19 08:16 debug_failed_EoSY_2025-26_Assessment_Results_attempt2.png
 84K -rw-r--r-- 1 root root  84K Mar 19 08:16 debug_failed_EoSY_2025-26_Assessment_Results_attempt3.png
4.0K drwxr-xr-x 2 root root 4.0K Mar 19 08:03 downloads
4.0K drwx------ 6 root root 4.0K Mar 19 08:02 drive
4.0K drwxr-xr-x 1 root root 4.0K Mar 17 17:58 sample_data


# 1.0. Automation

## 1.1. Single implementation

In [ ]:
# ─── CONFIG ───────────────────────────────────────────────
DASHBOARD_URL = (
    "https://lookerstudio.google.com/u/0/reporting/"
    "1a2b7706-a54d-4646-82b7-93da6a9eb7d3/page/p_3bjcc64yld?s=suPjwoowKzA"
)
DOWNLOAD_DIR = os.path.abspath("./downloads")
# ──────────────────────────────────────────────────────────

In [ ]:
async def export_list_of_schools():
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(accept_downloads=True)
        page = await context.new_page()

        # ── Step 1: Load the dashboard ────────────────────────────────────
        print("Step 1: Loading dashboard...")
        await page.goto(DASHBOARD_URL, wait_until="networkidle", timeout=90000)
        await asyncio.sleep(4)

        # ── Step 2: Enter the iframe ──────────────────────────────────────
        print("Step 2: Locating iframe...")
        frame = None
        for f in page.frames:
            print(f"  Found frame: {f.url}")
            if "lookerstudio.google.com" in f.url and f.url != DASHBOARD_URL:
                frame = f
                break
        if frame is None:
            print("  ⚠️ Falling back to main page")
            frame = page

        # ── Step 3: Click the EoSY 2025-26 tab ───────────────────────────
        print("Step 3: Clicking EoSY tab...")
        await frame.get_by_text("EoSY 2025-26 Assessment Results", exact=True).click()
        await asyncio.sleep(5)

        # ── Step 4: Right-click on a table data cell (NOT the title) ──────────
        # Confirmed: right-click on title only shows "Link to this component"
        # Must right-click on actual table row/cell to get the Export menu
        print("Step 4: Right-clicking on table data cell...")

        # Target a cell in the first data row instead of the title
        table_cell = frame.get_by_text("BARMM", exact=True).first
        await table_cell.scroll_into_view_if_needed()
        await table_cell.click(button="right")
        await asyncio.sleep(2)

        # Take screenshot to confirm correct menu appeared
        await page.screenshot(path="debug_right_click_menu.png")
        print("  Screenshot saved: debug_right_click_menu.png")

        # ── Step 5: Hover "Export chart..." to reveal submenu ────────────────
        print("Step 5: Hovering over 'Export chart...'...")
        await frame.get_by_text("Export chart...", exact=True).hover()
        await asyncio.sleep(1)

        # ── Step 6: Click "Export data" ───────────────────────────────────────
        print("Step 6: Clicking 'Export data'...")
        await frame.locator("button[data-test-id='Export data']").click()
        await asyncio.sleep(2)

        # ── Step 7: Select CSV and confirm ───────────────────────────────────
        print("Step 7: Selecting CSV format...")
        # Dialog overlays can appear on main page or frame — try both
        csv_input = page.locator("input#mat-radio-0-input")
        if await csv_input.count() == 0:
            csv_input = frame.locator("input#mat-radio-0-input")
        await csv_input.click()
        await asyncio.sleep(1)

        print("Step 7: Clicking Export...")
        export_btn = page.locator("mat-dialog-actions button.mat-mdc-raised-button")
        if await export_btn.count() == 0:
            export_btn = frame.locator("mat-dialog-actions button.mat-mdc-raised-button")

        async with page.expect_download() as download_info:
            await export_btn.click()

        # ── Step 8: Save the downloaded file ─────────────────────────────────
        download = await download_info.value

        # Use the original filename suggested by the browser
        original_filename = download.suggested_filename
        save_path = os.path.join(DOWNLOAD_DIR, original_filename)
        await download.save_as(save_path)
        print(f"✅ Done! File saved as: {original_filename}")
        print(f"   Full path: {save_path}")

        await browser.close()

In [ ]:
# await export_list_of_schools()

Step 1: Loading dashboard...
Step 2: Locating iframe...
  Found frame: https://lookerstudio.google.com/reporting/1a2b7706-a54d-4646-82b7-93da6a9eb7d3/page/p_3bjcc64yld?s=suPjwoowKzA
Step 3: Clicking EoSY tab...
Step 4: Right-clicking on table data cell...
  Screenshot saved: debug_right_click_menu.png
Step 5: Hovering over 'Export chart...'...
Step 6: Clicking 'Export data'...
Step 7: Selecting CSV format...
Step 7: Clicking Export...
✅ Done! File saved to: /content/list_of_schools.csv


## 1.2. OOP

In [53]:
class CRLADashboardExporter:
    """
    Automates CSV export of 'List of Schools' tables
    from the CRLA National Dashboard (SY 2025-26) and
    CRLA Results Archive (SY 2024-25) in Looker Studio.

    Standardized filename format:
        CRLA_{period}_{school_year}_{timestamp}.csv
    Example:
        CRLA_BoSY_2025-26_20260319_073739.csv
        CRLA_EoSY_2024-25_20260319_073833.csv
    """

    DASHBOARD_URL = (
        "https://lookerstudio.google.com/u/0/reporting/"
        "1a2b7706-a54d-4646-82b7-93da6a9eb7d3/page/p_3bjcc64yld?s=suPjwoowKzA"
    )
    ARCHIVE_URL = (
        "https://lookerstudio.google.com/u/0/reporting/"
        "27d41d23-fdc6-41c7-8c76-0c5d74e45911/page/p_3bjcc64yld?s=tpMlVXbOMxU"
    )
    TABS = [
        "BoSY 2025-26 Assessment Results",
        "MoSY 2025-26 Assessment Results",
        "EoSY 2025-26 Assessment Results",
    ]
    PERIODS = ["BoSY", "EoSY"]

    MAX_EXPORT_RETRIES = 3
    RETRY_DELAY = 3

    def __init__(self, download_dir: str = "./downloads", headless: bool = True):
        self.download_dir = os.path.abspath(download_dir)
        self.headless = headless
        os.makedirs(self.download_dir, exist_ok=True)
        self.results = []

    # ── Browser factory ───────────────────────────────────────────────────────

    async def _new_browser_context(self, playwright):
        browser = await playwright.chromium.launch(
            headless=self.headless,
            args=["--lang=en-US"]
        )
        context = await browser.new_context(
            accept_downloads=True,
            locale="en-US"
        )
        return browser, context

    # ── Filename builder ──────────────────────────────────────────────────────

    def _build_filename(self, period: str, school_year: str) -> str:
        """
        Build a standardized filename.
        Format: CRLA_{period}_{school_year}_{timestamp}.csv
        Example: CRLA_BoSY_2025-26_20260319_073739.csv
        """
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        return f"CRLA_{period}_{school_year}_{timestamp}.csv"

    # ── Overlay cleanup ───────────────────────────────────────────────────────

    async def _dismiss_overlays(self, page):
        """Dismiss any open menus, dialogs or backdrops."""
        try:
            await page.keyboard.press("Escape")
            await asyncio.sleep(1)
            await page.keyboard.press("Escape")
            await asyncio.sleep(1)
        except Exception:
            pass

    # ── Public entry points ───────────────────────────────────────────────────

    async def export_everything(self):
        """Export all CSVs from both dashboards in a single browser session."""
        self.results = []
        async with async_playwright() as p:
            browser, context = await self._new_browser_context(p)
            page = await context.new_page()
            try:
                # SY 2025-26
                print("\n" + "═" * 60)
                print("SY 2025-26 — CRLA National Dashboard")
                print("═" * 60)
                frame = await self._load_dashboard(page, self.DASHBOARD_URL)
                for tab_name in self.TABS:
                    await self._export_tab(page, frame, tab_name)

                # SY 2024-25
                print("\n" + "═" * 60)
                print("SY 2024-25 — CRLA Results Archive")
                print("═" * 60)
                await self._load_dashboard(page, self.ARCHIVE_URL)
                await asyncio.sleep(3)
                frame = page.frames[0]
                await frame.get_by_text(
                    "SY 2024-25 Assessment Results", exact=True
                ).click()
                await asyncio.sleep(8)
                frame = page.frames[0]
                for period in self.PERIODS:
                    await self._export_archive_period(page, frame, period)
            finally:
                await browser.close()
                self._print_summary()

    async def export_all(self):
        """Export SY 2025-26 tabs only."""
        self.results = []
        async with async_playwright() as p:
            browser, context = await self._new_browser_context(p)
            page = await context.new_page()
            try:
                frame = await self._load_dashboard(page, self.DASHBOARD_URL)
                for tab_name in self.TABS:
                    await self._export_tab(page, frame, tab_name)
            finally:
                await browser.close()
                self._print_summary()

    async def export_archive(self):
        """Export SY 2024-25 archive periods only."""
        self.results = []
        async with async_playwright() as p:
            browser, context = await self._new_browser_context(p)
            page = await context.new_page()
            try:
                frame = await self._load_dashboard(page, self.ARCHIVE_URL)
                print("Clicking 'SY 2024-25 Assessment Results' tab...")
                await frame.get_by_text(
                    "SY 2024-25 Assessment Results", exact=True
                ).click()
                await asyncio.sleep(6)
                for period in self.PERIODS:
                    await self._export_archive_period(page, frame, period)
            finally:
                await browser.close()
                self._print_summary()

    # ── Private helpers ───────────────────────────────────────────────────────

    async def _load_dashboard(self, page, url: str):
        """
        Navigate to a dashboard URL and return the content frame.
        Uses domcontentloaded instead of networkidle — Looker Studio
        makes continuous background requests that prevent networkidle
        from resolving reliably.
        Retries navigation up to 3 times on timeout.
        """
        print(f"\nLoading dashboard: {url}")

        for attempt in range(1, 4):
            try:
                if attempt > 1:
                    print(f"  ⟳ Navigation retry {attempt}/3...")
                    await asyncio.sleep(3)
                await page.goto(
                    url,
                    wait_until="domcontentloaded",
                    timeout=60000
                )
                break
            except Exception as e:
                print(f"  ⚠️ Navigation attempt {attempt} failed: {str(e)[:80]}")
                if attempt == 3:
                    raise Exception(
                        f"Could not load dashboard after 3 attempts: {url}"
                    )

        await asyncio.sleep(8)

        print("Locating frame...")
        if url == self.ARCHIVE_URL:
            frame = page.frames[0]
            print(f"  Using Frame 0 for archive: {frame.url}")
            return frame

        frame = None
        for f in page.frames:
            print(f"  Found frame: {f.url}")
            if "lookerstudio.google.com" in f.url and f.url != url:
                frame = f
                break
        if frame is None:
            print("  ⚠️ No child iframe found — falling back to main page")
            frame = page
        return frame

    async def _export_tab(self, page, frame, tab_name: str):
        """SY 2025-26: Click a tab then export with fault tolerance."""
        print(f"\n{'─' * 60}")
        print(f"Tab: {tab_name}")
        print(f"{'─' * 60}")

        match = re.match(r"(BoSY|EoSY|MoSY)\s+(\d{4}-\d{2})", tab_name)
        if match:
            period = match.group(1)
            school_year = match.group(2)
        else:
            period = tab_name.split()[0]
            school_year = "unknown"

        # Dismiss any lingering overlays before clicking the tab
        await self._dismiss_overlays(page)

        print("  Clicking tab...")
        await frame.get_by_text(tab_name, exact=True).click()
        await asyncio.sleep(5)

        await self._export_with_retry(
            page, frame,
            label=tab_name,
            period=period,
            school_year=school_year
        )

    async def _export_archive_period(self, page, frame, period: str):
        """SY 2024-25: Select a period from the dropdown then export."""
        print(f"\n{'─' * 60}")
        print(f"Assessment Period: {period}")
        print(f"{'─' * 60}")

        await self._select_assessment_period(page, frame, period)

        print(f"  Waiting for dashboard to reflect '{period}'...")
        try:
            live_frame = page.frames[0]
            await live_frame.wait_for_function(
                f"() => document.body.innerText.includes('{period} 2024-25')",
                timeout=15000
            )
            print(f"  Dashboard updated to '{period}' ✅")
        except Exception:
            print(f"  ⚠️ Could not confirm period change — waiting extra 5s...")
            await asyncio.sleep(5)

        await self._export_with_retry(
            page, page.frames[0],
            label=f"{period} 2024-25",
            period=period,
            school_year="2024-25"
        )

    async def _export_with_retry(
        self, page, frame, label: str, period: str, school_year: str
    ):
        """
        Retry wrapper around _export_table.
        Dismisses overlays before each retry for a clean state.
        Failures are recorded but do NOT crash the overall run.
        """
        for attempt in range(1, self.MAX_EXPORT_RETRIES + 1):
            try:
                if attempt > 1:
                    print(f"  ⟳ Retry attempt {attempt}/{self.MAX_EXPORT_RETRIES}...")
                    await asyncio.sleep(self.RETRY_DELAY)
                    await self._dismiss_overlays(page)

                save_path = await self._export_table(
                    page, frame, period, school_year
                )
                self.results.append({
                    "label": label,
                    "status": "✅ Success",
                    "file": os.path.basename(save_path)
                })
                return

            except Exception as e:
                print(f"  ⚠️ Attempt {attempt} failed: {type(e).__name__}: {str(e)[:100]}")
                safe_label = label.replace(' ', '_')
                await page.screenshot(
                    path=f"debug_failed_{safe_label}_attempt{attempt}.png"
                )
                await self._dismiss_overlays(page)

                if attempt == self.MAX_EXPORT_RETRIES:
                    print(f"  ❌ All {self.MAX_EXPORT_RETRIES} attempts failed for '{label}'")
                    self.results.append({
                        "label": label,
                        "status": "❌ Failed",
                        "file": f"debug_failed_{safe_label}_attempt{attempt}.png"
                    })

    async def _select_assessment_period(self, page, frame, period: str):
        """
        Click the Assessment Period dropdown and select BoSY or EoSY.
        Language-agnostic: targets the dropdown by its confirmed class
        'lego-control' instead of text — works in any UI language.
        Confirmed: span[title='BoSY'] / span[title='EoSY'] are always
        in English regardless of UI language.
        """
        live_frame = page.frames[0]

        # Wait for the lego-control buttons to render — these are the
        # filter dropdowns and are present in all UI languages
        print(f"  Waiting for filter dropdowns to render...")
        await live_frame.wait_for_selector(
            "button.lego-control", timeout=30000
        )
        print(f"  Filter dropdowns found in DOM ✅")

        # Click the last lego-control button — confirmed to be Assessment Period
        result = await live_frame.evaluate("""
            () => {
                const buttons = Array.from(
                    document.querySelectorAll('button.lego-control')
                );
                if (buttons.length === 0) return 'ERROR: No lego-control buttons found';
                // Assessment Period is the last filter button
                const btn = buttons[buttons.length - 1];
                btn.click();
                return 'Clicked lego-control button ' + (buttons.length - 1) +
                      ': ' + btn.textContent.trim().substring(0, 50);
            }
        """)
        print(f"  Dropdown click result: {result}")
        await asyncio.sleep(2)

        print(f"  Selecting '{period}'...")
        # span[title='BoSY'] / span[title='EoSY'] are always English
        # regardless of UI language — confirmed from DOM inspection
        option = live_frame.locator(f"span[title='{period}']").first
        if await option.count() == 0:
            option = page.locator(f"span[title='{period}']").first
        await option.wait_for(state="visible", timeout=10000)
        await option.click(force=True)
        await asyncio.sleep(3)

        await page.keyboard.press("Escape")
        await asyncio.sleep(2)
        try:
            await live_frame.locator("div.popup-backdrop").wait_for(
                state="hidden", timeout=5000
            )
            print(f"  Backdrop cleared ✅")
        except Exception:
            await page.mouse.click(100, 400)
            await asyncio.sleep(2)

    async def _export_table(
        self, page, frame, period: str, school_year: str
    ) -> str:
        """
        Core export logic — right-click → hover item 3 → click submenu
        first item → CSV → download.
        Fully language-agnostic: uses menu item position, not text.
        Confirmed: export trigger is always item index 3 in both
        5-item and 6-item context menus across all UI languages.
        """
        print("  Scrolling to List of Schools table...")
        table_title = frame.get_by_text("List of Schools", exact=True)
        await table_title.scroll_into_view_if_needed()
        await asyncio.sleep(1)

        print("  Right-clicking table data cell...")
        table_cell = frame.get_by_text("BARMM", exact=True).first
        await table_cell.scroll_into_view_if_needed()
        bbox = await table_cell.bounding_box()
        if bbox:
            await page.mouse.click(
                bbox["x"] + bbox["width"] / 2,
                bbox["y"] + bbox["height"] / 2,
                button="right"
            )
        else:
            await table_cell.click(button="right")
        await asyncio.sleep(2)

        menu_items = page.locator(
            '[role="menu"] button, '
            '.mat-mdc-menu-content button, '
            '.mat-menu-content button'
        )
        count = await menu_items.count()
        print(f"  Found {count} menu items")

        if count == 0:
            raise Exception("Context menu did not appear after right-click")

        # Require at least 4 items — fewer means wrong element was right-clicked
        if count < 4:
            raise Exception(
                f"Expected at least 4 menu items, got {count}. "
                "Right-click landed on wrong element — will retry."
            )

        bboxes = []
        for i in range(count):
            b = await menu_items.nth(i).bounding_box()
            if b:
                bboxes.append({
                    "x": b["x"] + b["width"] / 2,
                    "y": b["y"] + b["height"] / 2,
                    "index": i
                })

        # ── Position-based export trigger — fully language agnostic ───────
        # Confirmed: export submenu trigger is always item index 3
        # in both 5-item and 6-item menus across English, Dutch and Chinese
        print(f"  Hovering export trigger (item 3)...")
        export_trigger = bboxes[3]
        await page.mouse.move(export_trigger["x"], export_trigger["y"])
        await asyncio.sleep(2)  # Wait for submenu to render

        # Click the first visible item in the submenu — always "Export data"
        # regardless of UI language
        export_btn_info = await page.evaluate("""
            () => {
                const menus = Array.from(document.querySelectorAll('[role="menu"]'));
                // Submenu is the last opened menu
                const submenu = menus[menus.length - 1];
                if (!submenu) return { found: false };
                const items = Array.from(
                    submenu.querySelectorAll('button[role="menuitem"]')
                );
                const visible = items.filter(b => b.offsetParent !== null);
                if (visible.length > 0) {
                    visible[0].click();
                    return { found: true, text: visible[0].textContent.trim() };
                }
                return { found: false };
            }
        """)

        if not export_btn_info.get("found"):
            raise Exception("Export submenu did not appear after hovering item 3")

        print(f"  ✅ Clicked: '{export_btn_info.get('text')}'")
        await asyncio.sleep(2)

        # Select CSV — exact regex match to avoid matching "CSV (Excel)"
        print("  Selecting CSV format...")
        csv_radio = page.locator("mat-radio-button").filter(
            has=page.locator("label", has_text=re.compile(r"^CSV$"))
        ).first
        if await csv_radio.count() == 0:
            csv_radio = page.locator("mat-radio-button").first
        await csv_radio.wait_for(state="visible", timeout=10000)
        await csv_radio.click()
        await asyncio.sleep(1)

        print("  Clicking Export button...")
        async with page.expect_download(timeout=60000) as download_info:
            export_confirm = page.locator(
                "mat-dialog-actions button.mat-mdc-raised-button"
            )
            await export_confirm.wait_for(state="visible", timeout=10000)
            await export_confirm.click()

        download = await download_info.value
        filename = self._build_filename(period, school_year)
        save_path = os.path.join(self.download_dir, filename)
        await download.save_as(save_path)
        print(f"  ✅ Saved: {filename}")
        return save_path

    # ── Summary ───────────────────────────────────────────────────────────────

    def _print_summary(self):
        """Print a full run summary showing success/failure of each export."""
        print("\n" + "═" * 60)
        print("EXPORT RUN SUMMARY")
        print("═" * 60)
        for r in self.results:
            print(f"  {r['status']} {r['label']}")
            print(f"           {r['file']}")
        successes = sum(1 for r in self.results if "Success" in r["status"])
        failures = sum(1 for r in self.results if "Failed" in r["status"])
        print(f"\n  Total: {successes} succeeded, {failures} failed")
        if failures > 0:
            print(f"  ⚠️ Check debug screenshots for failed exports")
        print("═" * 60)

In [54]:
# ── Entry point ───────────────────────────────────────────────────────────────
exporter = CRLADashboardExporter(download_dir="./downloads", headless=True)

In [55]:
s_time = time.time()

# Uncomment the method you want to run:
await exporter.export_everything()   # Both dashboards
# await exporter.export_all()        # SY 2025-26 only
# await exporter.export_archive()    # SY 2024-25 only

e_time = time.time()
t_time = e_time - s_time
print(f"\nTotal time: {t_time:,.2f} seconds")


════════════════════════════════════════════════════════════
SY 2025-26 — CRLA National Dashboard
════════════════════════════════════════════════════════════

Loading dashboard: https://lookerstudio.google.com/u/0/reporting/1a2b7706-a54d-4646-82b7-93da6a9eb7d3/page/p_3bjcc64yld?s=suPjwoowKzA
Locating frame...
  Found frame: https://lookerstudio.google.com/reporting/1a2b7706-a54d-4646-82b7-93da6a9eb7d3/page/p_3bjcc64yld?s=suPjwoowKzA

────────────────────────────────────────────────────────────
Tab: BoSY 2025-26 Assessment Results
────────────────────────────────────────────────────────────
  Clicking tab...
  Scrolling to List of Schools table...
  Right-clicking table data cell...
  Found 5 menu items
  Hovering export trigger (item 3)...
  ✅ Clicked: '匯出資料'
  Selecting CSV format...
  Clicking Export button...
  ✅ Saved: CRLA_BoSY_2025-26_20260319_090545.csv

────────────────────────────────────────────────────────────
Tab: MoSY 2025-26 Assessment Results
──────────────────────────

In [47]:
!ls -lhsa ./downloads/

total 36M
4.0K drwxr-xr-x 2 root root 4.0K Mar 19 08:45 .
4.0K drwxr-xr-x 1 root root 4.0K Mar 19 08:44 ..
8.5M -rw-r--r-- 1 root root 8.5M Mar 19 08:44 CRLA_BoSY_2024-25_20260319_084430.csv
9.7M -rw-r--r-- 1 root root 9.7M Mar 19 08:41 CRLA_BoSY_2025-26_20260319_084121.csv
8.7M -rw-r--r-- 1 root root 8.7M Mar 19 08:45 CRLA_EoSY_2024-25_20260319_084509.csv
8.7M -rw-r--r-- 1 root root 8.7M Mar 19 08:41 CRLA_EoSY_2025-26_20260319_084150.csv


In [52]:
# !rm -rf downloads/*

In [41]:
# !rm -f debug_*